In [ ]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
import scienceplots as _  # noqa: F401
from atab10 import atab10

plt.style.use(["science", "nature"])

width = 390
aspect = 1 + 2**0.5
scale = 1.2 / 72
figsize = (width * scale, (width * scale) / aspect)

In [ ]:
df_evaluate = pd.read_csv("../logs/evaluate/data_frame.csv")
df_dd_run = pd.read_csv("../logs/dd_run/data_frame.csv")

cmap = {
    "FT": atab10(0),
    "EWC": atab10(1),
    "SI": atab10(2),
    "iCaRL": atab10(3),
    "ER": atab10(4),
    "LWF": atab10(5),
}


def filters(df: pd.DataFrame) -> pd.DataFrame:
    mask = ~df.detector_label.isin(["oracle", "ADWIN_JOINT"])
    return df[mask]


df_evaluate = filters(df_evaluate)
df_dd_run = filters(df_dd_run)

fig, (ax_true, ax_sim) = plt.subplots(
    1, 2, figsize=figsize, sharey=True, constrained_layout=True
)
axies = [ax_true, ax_sim]

order = (
    df_evaluate.groupby("detector_label")["dd.wasserstein_distance"]
    .median()
    .sort_values()
    .index
)

sns.boxplot(
    data=df_evaluate,
    hue="strategy",
    y="dd.wasserstein_distance",
    x="detector_label",
    ax=ax_true,
    palette=cmap,
    legend=False,
    order=order,
)
ax_true.set_title("True Performance")
ax_true.set_xlabel("")
ax_true.set_ylabel("Wasserstein Distance")

sns.boxplot(
    data=df_dd_run,
    hue="strategy",
    y="dd.wasserstein_distance",
    x="detector_label",
    ax=ax_sim,
    palette=cmap,
    legend=True,
    order=order,
)
ax_sim.legend_.set_title("Strategy")
# Number of cols

ax_sim.set_title("Simulated Performance")
ax_sim.set_ylabel(r"Wasserstein Distance $\downarrow$")

for ax in axies:
    ax.xaxis.set_minor_locator(plt.NullLocator())
    ax.set_xlabel("")
    ax.set_ylim(-0.1, 5.0)

# Save figure
fig.savefig("../fig/detector_performance.pdf")